*******# Setup*******

In [ ]:
# =====================================================================
# CELL 1 — COMPLETE TRELLIS SETUP  (idempotent; safe to re-run)
# Requirements: Kaggle Internet ON, GPU T4 x2 (or any T4)
# =====================================================================
import os, sys, subprocess, shutil, time, textwrap

ROOT  = "/kaggle/working"
VENV  = f"{ROOT}/trellis_env"
PY    = f"{VENV}/bin/python"
REPO  = f"{ROOT}/TRELLIS"
EXT   = f"{ROOT}/extensions"
CUDA  = "/usr/local/cuda-11.8"
HF    = f"{ROOT}/hf_cache"
CONSTRAINTS = f"{ROOT}/constraints.txt"
os.makedirs(EXT, exist_ok=True)

# ---- environment used for every subprocess (build + runtime) ----
ENV = os.environ.copy()
ENV.update({
    "CUDA_HOME": CUDA,
    "CUDA_PATH": CUDA,
    "PATH": f"{VENV}/bin:{CUDA}/bin:" + ENV.get("PATH", ""),
    "LD_LIBRARY_PATH": f"{CUDA}/lib64:" + ENV.get("LD_LIBRARY_PATH", ""),
    "TORCH_CUDA_ARCH_LIST": "7.5",          # Tesla T4
    "MAX_JOBS": "4",
    "FORCE_CUDA": "1",
    "NVCC_APPEND_FLAGS": "-allow-unsupported-compiler",
    "ATTN_BACKEND": "xformers",
    "SPCONV_ALGO": "native",
    "HF_HOME": HF,
    "PYTHONPATH": REPO,
    "PIP_DISABLE_PIP_VERSION_CHECK": "1",
})

def tail(s, n=150):
    lines = (s or "").strip().splitlines()
    return "\n".join(lines[-n:])

def sh(cmd, desc="", cwd=None, check=True):
    """Run a command; on failure print the REAL stdout/stderr and raise."""
    shell = isinstance(cmd, str)
    t0 = time.time()
    if desc: print(f"\n>>> {desc}", flush=True)
    p = subprocess.run(cmd, shell=shell, cwd=cwd, env=ENV,
                       stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if p.returncode != 0 and check:
        print("\n" + "!" * 78)
        print(f"COMMAND FAILED (exit {p.returncode}): {cmd if shell else ' '.join(cmd)}")
        print("----- STDOUT (tail) -----\n" + tail(p.stdout))
        print("----- STDERR (tail) -----\n" + tail(p.stderr))
        print("!" * 78)
        raise RuntimeError(f"Step failed: {desc or cmd}")
    if desc: print(f"    done in {time.time()-t0:.0f}s", flush=True)
    return p

def py_ok(code):
    """True if `code` runs without error inside the TRELLIS venv."""
    return subprocess.run([PY, "-c", code], env=ENV, capture_output=True, text=True).returncode == 0

def uv_install(args, desc, extra=None):
    cmd = ["uv", "pip", "install", "--python", PY, "-c", CONSTRAINTS,
           "--index-strategy", "unsafe-best-match"] + args + (extra or [])
    sh(cmd, desc)

print("=" * 78, "\nTRELLIS SETUP (Python 3.10 / torch 2.4.0+cu118 / T4)\n" + "=" * 78)

# ---------------------------------------------------------------- 1. CUDA 11.8
if os.path.exists(f"{CUDA}/bin/nvcc"):
    print(f"[1/12] CUDA 11.8 toolkit already present: {CUDA}")
else:
    print("[1/12] Installing CUDA Toolkit 11.8 (toolkit only, ~4 GB download)...")
    run_file = "/tmp/cuda_11.8.0_520.61.05_linux.run"
    if not os.path.exists(run_file):
        sh(["wget", "-q", "-c", "-O", run_file,
            "https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run"],
           "Downloading CUDA 11.8 installer")
    sh(["sh", run_file, "--silent", "--toolkit", f"--toolkitpath={CUDA}",
        "--no-opengl-libs", "--no-drm", "--no-man-page", "--override"],
       "Installing CUDA 11.8 toolkit")
    if os.path.exists(run_file): os.remove(run_file)
assert os.path.exists(f"{CUDA}/bin/nvcc"), "nvcc 11.8 not found after install"
print(sh([f"{CUDA}/bin/nvcc", "--version"]).stdout.strip().splitlines()[-1])

# ---------------------------------------------------------------- 2. Python 3.10 venv (uv)
if os.path.exists(PY) and py_ok("import sys; assert sys.version_info[:2]==(3,10)"):
    print("[2/12] Python 3.10 venv already exists")
else:
    print("[2/12] Creating Python 3.10 venv with uv (no ensurepip)...")
    sh([sys.executable, "-m", "pip", "install", "-q", "-U", "uv"], "Installing uv")
    if os.path.exists(VENV): shutil.rmtree(VENV)
    sh(["uv", "venv", "--python", "3.10", "--seed", VENV], "Creating venv (Python 3.10)")
assert os.path.exists(PY)
print(sh([PY, "--version"]).stdout.strip())

# constraints: nothing later may change these
with open(CONSTRAINTS, "w") as f:
    f.write("torch==2.4.0\ntorchvision==0.19.0\nnumpy<2\ntransformers==4.50.1\n")

# ---------------------------------------------------------------- 3. Build tooling
if py_ok("import setuptools, wheel, ninja"):
    print("[3/12] Build tooling present")
else:
    uv_install(["setuptools<70", "wheel", "ninja", "packaging", "numpy<2", "cmake"],
               "[3/12] Installing build tooling")

# ---------------------------------------------------------------- 4. PyTorch
if py_ok("import torch,torchvision; assert torch.__version__.startswith('2.4.0') "
         "and torch.version.cuda=='11.8' and torchvision.__version__.startswith('0.19.0')"):
    print("[4/12] torch 2.4.0+cu118 / torchvision 0.19.0 already installed")
else:
    uv_install(["torch==2.4.0+cu118", "torchvision==0.19.0+cu118"],
               "[4/12] Installing PyTorch 2.4.0 + torchvision 0.19.0 (cu118)",
               extra=["--index-url", "https://download.pytorch.org/whl/cu118",
                      "--extra-index-url", "https://pypi.org/simple"])

# ---------------------------------------------------------------- 5. TRELLIS repo
if os.path.exists(f"{REPO}/trellis/__init__.py"):
    print("[5/12] TRELLIS repo already cloned")
    sh(["git", "submodule", "update", "--init", "--recursive"], "Syncing submodules", cwd=REPO, check=False)
else:
    if os.path.exists(REPO): shutil.rmtree(REPO)
    sh(["git", "clone", "--recurse-submodules", "https://github.com/microsoft/TRELLIS.git", REPO],
       "[5/12] Cloning official TRELLIS (with submodules)")

# ---------------------------------------------------------------- 6. Basic deps + transformers
if py_ok("import plyfile, trimesh, utils3d, rembg, onnxruntime, open3d, xatlas, pyvista, pymeshfix, igraph, "
         "imageio, easydict, cv2, scipy, transformers; assert transformers.__version__=='4.50.1'"):
    print("[6/12] TRELLIS basic deps already installed")
else:
    uv_install(["pillow", "imageio", "imageio-ffmpeg", "tqdm", "easydict", "opencv-python-headless",
                "scipy", "ninja", "rembg", "onnxruntime", "trimesh", "open3d", "xatlas", "pyvista",
                "pymeshfix", "igraph", "plyfile", "einops", "psutil", "safetensors", "huggingface_hub",
                "transformers==4.50.1",
                "git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8"],
               "[6/12] Installing TRELLIS basic dependencies + transformers 4.50.1 + plyfile + trimesh + utils3d")

# ---------------------------------------------------------------- 7. xformers
if py_ok("import xformers; assert xformers.__version__.startswith('0.0.27.post2')"):
    print("[7/12] xformers 0.0.27.post2 already installed")
else:
    uv_install(["xformers==0.0.27.post2", "--no-deps"],
               "[7/12] Installing xformers 0.0.27.post2 (cu118)",
               extra=["--index-url", "https://download.pytorch.org/whl/cu118"])

# ---------------------------------------------------------------- 8. spconv
if py_ok("import spconv.pytorch"):
    print("[8/12] spconv already installed")
else:
    uv_install(["spconv-cu118"], "[8/12] Installing spconv-cu118")

# ---------------------------------------------------------------- 9. nvdiffrast
if py_ok("import nvdiffrast.torch"):
    print("[9/12] nvdiffrast already installed")
else:
    d = f"{EXT}/nvdiffrast"
    if not os.path.exists(d):
        sh(["git", "clone", "https://github.com/NVlabs/nvdiffrast.git", d], "Cloning nvdiffrast")
    uv_install([d, "--no-build-isolation"], "[9/12] Installing nvdiffrast")

# ---------------------------------------------------------------- 10. Kaolin 0.16.0
if py_ok("import kaolin; assert kaolin.__version__=='0.16.0'"):
    print("[10/12] Kaolin 0.16.0 already installed")
else:
    uv_install(["kaolin==0.16.0"], "[10/12] Installing Kaolin 0.16.0 (torch-2.4.0_cu118)",
               extra=["-f", "https://nvidia-kaolin.s3.us-east-2.amazonaws.com/torch-2.4.0_cu118.html"])

# ---------------------------------------------------------------- 11. diffoctreerast + mip-splatting
if py_ok("import diffoctreerast"):
    print("[11a/12] diffoctreerast already installed")
else:
    d = f"{EXT}/diffoctreerast"
    if not os.path.exists(d):
        sh(["git", "clone", "--recurse-submodules", "https://github.com/JeffreyXiang/diffoctreerast.git", d],
           "Cloning diffoctreerast")
    uv_install([d, "--no-build-isolation"], "[11a/12] Compiling diffoctreerast (CUDA, a few minutes)")

if py_ok("import diff_gaussian_rasterization"):
    print("[11b/12] diff-gaussian-rasterization already installed")
else:
    d = f"{EXT}/mip-splatting"
    if not os.path.exists(d):
        sh(["git", "clone", "--recursive", "https://github.com/autonomousvision/mip-splatting.git", d],
           "Cloning mip-splatting (recursive, includes glm)")
    sub = f"{d}/submodules/diff-gaussian-rasterization"
    sh(["git", "submodule", "update", "--init", "--recursive"], "Syncing rasterizer submodules", cwd=sub, check=False)
    uv_install([sub, "--no-build-isolation"], "[11b/12] Compiling diff-gaussian-rasterization (CUDA, a few minutes)")

# ---------------------------------------------------------------- 12. Verify + import TRELLIS
print("\n[12/12] Verifying environment...")
verify = textwrap.dedent(r'''
import os, importlib
import torch, torchvision, transformers, xformers, trimesh
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
print("Torchvision:", torchvision.__version__)
print("Transformers:", transformers.__version__)
print("XFormers:", xformers.__version__)
import spconv.pytorch;            print("spconv: OK")
import nvdiffrast.torch;          print("nvdiffrast: OK")
import kaolin;                    print("Kaolin:", kaolin.__version__)
print("Trimesh:", trimesh.__version__)
import diffoctreerast;            print("diffoctreerast: OK")
import diff_gaussian_rasterization; print("diff-gaussian-rasterization: OK")
import plyfile, utils3d;          print("plyfile / utils3d: OK")
assert torch.__version__.startswith("2.4.0") and torch.version.cuda == "11.8" and torch.cuda.is_available()
from trellis.pipelines import TrellisImageTo3DPipeline
from trellis.utils import postprocessing_utils
print("ATTN_BACKEND =", os.environ.get("ATTN_BACKEND"), "| SPCONV_ALGO =", os.environ.get("SPCONV_ALGO"))
print("TRELLIS import: OK")
''')
p = subprocess.run([PY, "-c", verify], env=ENV, capture_output=True, text=True)
print(p.stdout)
if p.returncode != 0:
    print("!" * 78, "\nVERIFICATION FAILED\n----- STDERR -----\n" + tail(p.stderr, 200), "\n" + "!" * 78)
    raise RuntimeError("TRELLIS verification failed (see stderr above)")
print("SETUP COMPLETE")

***************# Project***************

In [ ]:
# =====================================================================
# CELL 2 — IMAGE → TRELLIS-image-large → MESH → GLB  (low-disk version)
# Cell 1 is NOT modified. No script file is written; code runs via `python -c`.
# Model files are read from /kaggle/working/hf_cache when present; anything
# missing is downloaded to /tmp (NOT to /kaggle/working).
# =====================================================================
import os, sys, shutil, subprocess, glob

ROOT   = "/kaggle/working"
VENV   = f"{ROOT}/trellis_env"          # underscore = the ONE used by Cell 1 (kept)
DUPENV = f"{ROOT}/trellis-env"          # hyphen     = stale duplicate (removed if safe)
PYTHON = f"{VENV}/bin/python"
REPO   = f"{ROOT}/TRELLIS"
CUDA   = "/usr/local/cuda-11.8"
OUT_DIR = f"{ROOT}/trellis_output"

BAKE_TEXTURED_GLB = False      # optional 1024 texture (needs gaussian decoder; off by default)
MIN_FREE_GB       = 0.3

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

def free_gb(path=ROOT):
    return shutil.disk_usage(path).free / 1024**3

def rm(path):
    if os.path.isdir(path) and not os.path.islink(path):
        shutil.rmtree(path, ignore_errors=True)
    elif os.path.lexists(path):
        try: os.remove(path)
        except OSError: pass

if not os.path.exists(PYTHON) or not os.path.exists(f"{REPO}/trellis"):
    raise RuntimeError("TRELLIS env/repo not found. Cell 1 must have been run in this session.")

# ---------------------------------------------------------------- 1. disk status + safe cleanup
print("=== DISK /kaggle/working (before) ===")
print(sh(f"df -h {ROOT}").stdout)

# (a) leftovers from previous failed/test runs (regenerated by this cell)
rm(f"{ROOT}/run_image_to_3d.py")
rm(OUT_DIR)
rm(f"{ROOT}/extensions")             # build sources (compiled packages already installed)
for p in ["/root/.cache/uv", "/root/.cache/pip", "/tmp/uv-cache"]:
    rm(p)
for g in glob.glob(f"{ROOT}/hf_cache/**/*.incomplete", recursive=True):
    rm(g)
for g in glob.glob(f"{REPO}/**/__pycache__", recursive=True):
    rm(g)

# (b) the duplicate 8.5 GB venv "trellis-env" (hyphen) -- only if trellis_env is verified healthy
if os.path.isdir(DUPENV) and os.path.realpath(DUPENV) != os.path.realpath(VENV):
    dup_real = os.path.realpath(DUPENV)
    py_real  = os.path.realpath(PYTHON)
    cfg_txt  = ""
    try:
        cfg_txt = open(f"{VENV}/pyvenv.cfg").read()
    except OSError:
        pass
    independent = (not py_real.startswith(dup_real + os.sep)) and ("trellis-env" not in cfg_txt)

    env_chk = os.environ.copy()
    env_chk.update({"CUDA_HOME": CUDA, "CUDA_VISIBLE_DEVICES": "0",
                    "LD_LIBRARY_PATH": f"{CUDA}/lib64:" + env_chk.get("LD_LIBRARY_PATH", "")})
    chk = subprocess.run(
        [PYTHON, "-c",
         "import torch, xformers, spconv.pytorch, nvdiffrast.torch, diffoctreerast, diff_gaussian_rasterization; "
         "print('ok')"],
        env=env_chk, capture_output=True, text=True)

    if independent and chk.returncode == 0:
        rm(DUPENV)
        print("Removed duplicate venv /kaggle/working/trellis-env (trellis_env verified healthy and independent).")
    else:
        print("KEPT trellis-env: safety check did not pass.")
        print("  independent of trellis_env:", independent)
        print("  trellis_env import check  :", "ok" if chk.returncode == 0 else chk.stderr[-400:])

print("\n=== DISK /kaggle/working (after) ===")
print(sh(f"df -h {ROOT}").stdout)
print("Largest items:")
print(sh(f"du -sh {ROOT}/* {ROOT}/.[!.]* 2>/dev/null | sort -h | tail -6").stdout)
print(f"Free in /kaggle/working: {free_gb():.2f} GB")
print(f"Free in /tmp (used for missing model files): {free_gb('/tmp'):.1f} GB")

if free_gb() < MIN_FREE_GB:
    raise RuntimeError(f"Only {free_gb():.2f} GB free in /kaggle/working - not enough to save the GLB. "
                       "See the listing above (do not delete TRELLIS, trellis_env or hf_cache).")
if free_gb("/tmp") < 6:
    raise RuntimeError("Less than 6 GB free in /tmp - cannot stage missing model files.")
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------------------------------------------------------- 2. inference code (passed via -c, no file on disk)
CODE = r'''
import os, sys, json, gc
os.environ["CUDA_HOME"] = "/usr/local/cuda-11.8"
os.environ["ATTN_BACKEND"] = "xformers"
os.environ["SPCONV_ALGO"] = "native"
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
sys.path.insert(0, "/kaggle/working/TRELLIS")

import numpy as np, torch, trimesh
from PIL import Image
from huggingface_hub import hf_hub_download
from trellis.pipelines import TrellisImageTo3DPipeline

IMAGE_PATH = "/kaggle/input/datasets/igustav/trellis-test-image/images.jpg"
OUT_GLB    = "/kaggle/working/trellis_output/trellis_result.glb"
OUT_TEX    = "/kaggle/working/trellis_output/trellis_result_textured_1024.glb"
BAKE       = os.environ.get("BAKE_TEXTURED_GLB", "0") == "1"

REPO_ID    = "microsoft/TRELLIS-image-large"
WORK_CACHE = "/kaggle/working/hf_cache/hub"     # existing cache (read-only use)
TMP_CACHE  = "/tmp/hf_dl"                       # downloads for missing files go here
LOCAL      = "/tmp/trellis_local"               # small local pipeline folder (symlinks)

# ---- 1. image ----
image = Image.open(IMAGE_PATH).convert("RGB")
print("Image loaded")
print(f"Input image : {IMAGE_PATH}")
print(f"Image size  : {image.size[0]} x {image.size[1]}", flush=True)

# ---- 2. build a local pipeline folder: reuse cache, download only what is missing (to /tmp) ----
def fetch(fn):
    try:
        return hf_hub_download(REPO_ID, fn, cache_dir=WORK_CACHE, local_files_only=True), "cache"
    except Exception:
        return hf_hub_download(REPO_ID, fn, cache_dir=TMP_CACHE), "downloaded to /tmp"

cfg_path, _ = fetch("pipeline.json")
cfg = json.load(open(cfg_path))
skip = {"slat_decoder_rf"} | (set() if BAKE else {"slat_decoder_gs"})   # not needed for mesh output
models = {k: v for k, v in cfg["args"]["models"].items() if k not in skip}
cfg["args"]["models"] = models
print("Skipping decoders:", sorted(skip), flush=True)

os.makedirs(f"{LOCAL}/ckpts", exist_ok=True)
json.dump(cfg, open(f"{LOCAL}/pipeline.json", "w"))
for name, rel in models.items():
    for ext in (".json", ".safetensors"):
        src, how = fetch(rel + ext)
        dst = f"{LOCAL}/{rel}{ext}"
        if os.path.lexists(dst):
            os.remove(dst)
        os.symlink(os.path.realpath(src), dst)
        if ext == ".safetensors":
            print(f"  {name:28s} <- {how}", flush=True)

# ---- 3. pipeline ----
print("GPU         :", torch.cuda.get_device_name(0), flush=True)
pipeline = TrellisImageTo3DPipeline.from_pretrained(LOCAL)
pipeline.cuda()
torch.cuda.empty_cache()
print("TRELLIS pipeline loaded")
print(f"GPU mem after load: {torch.cuda.memory_allocated()/1024**3:.2f} GB", flush=True)

# ---- 4. image -> 3D ----
formats = ["mesh", "gaussian"] if BAKE else ["mesh"]
outputs = pipeline.run(
    image,
    seed=42,
    formats=formats,
    sparse_structure_sampler_params={"steps": 12, "cfg_strength": 7.5},
    slat_sampler_params={"steps": 12, "cfg_strength": 3.0},
)
mesh = outputs["mesh"][0]
print("3D mesh generated", flush=True)

# ---- 5. export raw GLB ----
verts = mesh.vertices.detach().float().cpu().numpy()
faces = mesh.faces.detach().cpu().numpy()
n_v, n_f = len(verts), len(faces)
verts = verts @ np.array([[1, 0, 0], [0, 0, -1], [0, 1, 0]], dtype=np.float32)   # Z-up -> Y-up (glTF)

colors = None
va = getattr(mesh, "vertex_attrs", None)
if va is not None:
    va = va.detach().float().cpu().numpy()
    if va.ndim == 2 and va.shape[0] == n_v and va.shape[1] >= 3:
        colors = (np.clip(va[:, :3], 0, 1) * 255).astype(np.uint8)
    del va

trimesh.Trimesh(vertices=verts, faces=faces, vertex_colors=colors, process=False).export(OUT_GLB)
del verts, faces, colors
print("GLB exported successfully", flush=True)

print("\n===== RESULT =====")
print(f"Vertices   : {n_v:,}")
print(f"Faces      : {n_f:,}")
print(f"Output     : {OUT_GLB}")
print(f"GLB size   : {os.path.getsize(OUT_GLB)/1024**2:.1f} MB")
print(f"GPU alloc  : {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"GPU peak   : {torch.cuda.max_memory_allocated()/1024**3:.2f} GB", flush=True)

# ---- 6. optional textured GLB (disabled by default) ----
if BAKE:
    try:
        from trellis.utils import postprocessing_utils
        torch.cuda.empty_cache(); gc.collect()
        glb = postprocessing_utils.to_glb(outputs["gaussian"][0], mesh, simplify=0.95, texture_size=1024)
        glb.export(OUT_TEX)
        print(f"Textured GLB: {OUT_TEX} ({os.path.getsize(OUT_TEX)/1024**2:.1f} MB)")
    except Exception as e:
        print("Textured GLB skipped (non-fatal):", repr(e))
'''

# ---------------------------------------------------------------- 3. run inside the TRELLIS venv
env = os.environ.copy()
env.update({
    "CUDA_HOME": CUDA,
    "PATH": f"{VENV}/bin:{CUDA}/bin:" + env.get("PATH", ""),
    "LD_LIBRARY_PATH": f"{CUDA}/lib64:" + env.get("LD_LIBRARY_PATH", ""),
    "ATTN_BACKEND": "xformers",
    "SPCONV_ALGO": "native",
    "HF_HOME": f"{ROOT}/hf_cache",
    "PYTHONPATH": REPO,
    "CUDA_VISIBLE_DEVICES": "0",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "TORCH_CUDA_ARCH_LIST": "7.5",
    "PYTHONUNBUFFERED": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "BAKE_TEXTURED_GLB": "1" if BAKE_TEXTURED_GLB else "0",
})

proc = subprocess.Popen([PYTHON, "-u", "-c", CODE], env=env, cwd=REPO,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Image→3D failed with exit code {rc} (full traceback printed above)")

print(f"\nFree disk after run: {free_gb():.2f} GB")

In [ ]:
# ============================================================
# CELL 3 — TRELLIS GLB INTERACTIVE VIEWER
# ============================================================

import sys
import subprocess
import os

# ------------------------------------------------------------
# Install viewer dependencies if missing
# ------------------------------------------------------------

def install_if_missing(package, import_name=None):
    import_name = import_name or package

    try:
        __import__(import_name)
        print(f"[OK] {package} already installed.")
    except ImportError:
        print(f"[INSTALL] Installing {package}...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            package
        ])
        print(f"[OK] {package} installed.")


install_if_missing("trimesh")
install_if_missing("plotly")


# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

import numpy as np
import trimesh
import plotly.graph_objects as go


# ------------------------------------------------------------
# GLB path
# ------------------------------------------------------------

GLB_PATH = "/kaggle/working/trellis_output/trellis_result.glb"

if not os.path.exists(GLB_PATH):
    raise FileNotFoundError(
        f"GLB not found:\n{GLB_PATH}\n\n"
        "Make sure Cell 2 finished successfully."
    )


print()
print("=" * 60)
print("Loading TRELLIS 3D model...")
print("=" * 60)


# ------------------------------------------------------------
# Load GLB
# ------------------------------------------------------------

scene_or_mesh = trimesh.load(
    GLB_PATH,
    force="scene"
)


# ------------------------------------------------------------
# Extract mesh
# ------------------------------------------------------------

if isinstance(scene_or_mesh, trimesh.Scene):

    geometries = [
        g
        for g in scene_or_mesh.geometry.values()
        if isinstance(g, trimesh.Trimesh)
    ]

    if not geometries:
        raise RuntimeError(
            "No mesh geometry found inside GLB."
        )

    mesh = trimesh.util.concatenate(geometries)

else:

    mesh = scene_or_mesh


print("3D model loaded successfully.")
print(f"Vertices : {len(mesh.vertices):,}")
print(f"Faces    : {len(mesh.faces):,}")


# ------------------------------------------------------------
# Convert geometry to NumPy
# ------------------------------------------------------------

vertices = np.asarray(mesh.vertices)
faces = np.asarray(mesh.faces)

x = vertices[:, 0]
y = vertices[:, 1]
z = vertices[:, 2]

i = faces[:, 0]
j = faces[:, 1]
k = faces[:, 2]


# ------------------------------------------------------------
# Try to extract vertex colors
# ------------------------------------------------------------

vertex_colors = None

try:

    colors = mesh.visual.vertex_colors

    if (
        colors is not None
        and len(colors) == len(vertices)
    ):
        vertex_colors = colors[:, :3]

except Exception:
    pass


# ------------------------------------------------------------
# Create Plotly 3D viewer
# ------------------------------------------------------------

if vertex_colors is not None:

    print("Vertex colors detected.")

    fig = go.Figure(
        data=[
            go.Mesh3d(
                x=x,
                y=y,
                z=z,
                i=i,
                j=j,
                k=k,
                vertexcolor=vertex_colors,
                flatshading=False,
                opacity=1.0
            )
        ]
    )

else:

    print("No vertex colors detected.")
    print("Displaying mesh in light gray.")

    fig = go.Figure(
        data=[
            go.Mesh3d(
                x=x,
                y=y,
                z=z,
                i=i,
                j=j,
                k=k,
                color="lightgray",
                flatshading=False,
                opacity=1.0
            )
        ]
    )


# ------------------------------------------------------------
# Viewer layout
# ------------------------------------------------------------

fig.update_layout(

    title="TRELLIS Interactive 3D Mesh",

    scene=dict(

        xaxis=dict(
            visible=False
        ),

        yaxis=dict(
            visible=False
        ),

        zaxis=dict(
            visible=False
        ),

        aspectmode="data",

        bgcolor="white",

        camera=dict(
            eye=dict(
                x=1.6,
                y=1.6,
                z=1.1
            )
        )
    ),

    paper_bgcolor="white",

    margin=dict(
        l=0,
        r=0,
        t=40,
        b=0
    ),

    height=800
)


# ------------------------------------------------------------
# Save HTML viewer
# ------------------------------------------------------------

HTML_PATH = (
    "/kaggle/working/"
    "trellis_output/"
    "trellis_viewer.html"
)

fig.write_html(
    HTML_PATH,
    full_html=True,
    include_plotlyjs=True
)


# ------------------------------------------------------------
# Display viewer directly
# ------------------------------------------------------------

print()
print("=" * 60)
print("Viewer saved successfully:")
print(HTML_PATH)
print("=" * 60)
print()

fig.show()